In [1]:
import polars as pl
import json
from pathlib import Path

# Create path and ensure that it is correct
data_path = Path("../../data/raw/statsbomb/data")
print(data_path.exists())

for item in data_path.iterdir():
    print(item)

True
../../data/raw/statsbomb/data/lineups
../../data/raw/statsbomb/data/three-sixty
../../data/raw/statsbomb/data/matches
../../data/raw/statsbomb/data/competitions.json
../../data/raw/statsbomb/data/events


In [2]:
# Explore the competitions file
competition_path = Path("../../data/raw/statsbomb/data/competitions.json")

with open(competition_path) as comp:
    competitions = json.load(comp)

print(f"Total number of competitions Incl duplicates: {len(competitions)}")
print(competitions[0])

Total number of competitions Incl duplicates: 75
{'competition_id': 9, 'season_id': 281, 'country_name': 'Germany', 'competition_name': '1. Bundesliga', 'competition_gender': 'male', 'competition_youth': False, 'competition_international': False, 'season_name': '2023/2024', 'match_updated': '2024-09-28T20:46:38.893391', 'match_updated_360': '2025-07-06T04:26:07.636270', 'match_available_360': '2025-07-06T04:26:07.636270', 'match_available': '2024-09-28T20:46:38.893391'}


In [3]:
# Get the data for the UCL competition

for season in competitions:
    if season['competition_name'] == "Champions League":
        print(season)


{'competition_id': 16, 'season_id': 4, 'country_name': 'Europe', 'competition_name': 'Champions League', 'competition_gender': 'male', 'competition_youth': False, 'competition_international': False, 'season_name': '2018/2019', 'match_updated': '2025-05-08T15:10:50.835274', 'match_updated_360': '2021-06-13T16:17:31.694', 'match_available_360': None, 'match_available': '2025-05-08T15:10:50.835274'}
{'competition_id': 16, 'season_id': 1, 'country_name': 'Europe', 'competition_name': 'Champions League', 'competition_gender': 'male', 'competition_youth': False, 'competition_international': False, 'season_name': '2017/2018', 'match_updated': '2024-02-13T02:35:28.134882', 'match_updated_360': '2021-06-13T16:17:31.694', 'match_available_360': None, 'match_available': '2024-02-13T02:35:28.134882'}
{'competition_id': 16, 'season_id': 2, 'country_name': 'Europe', 'competition_name': 'Champions League', 'competition_gender': 'male', 'competition_youth': False, 'competition_international': False, '

In [4]:
# Get UCL id and season id
ucl_id = None
ucl_season_id = None
for season in competitions:
    if season['competition_name'] == "Champions League" and season['season_name'] == '2018/2019':
        ucl_id = season['competition_id']
        ucl_season_id = season['season_id']
        break
print(f"UCL ID:{ucl_id}\nUCL season id:{ucl_season_id}")

UCL ID:16
UCL season id:4


In [5]:
# Look at UCL  matches
matches_path = Path('../../data/raw/statsbomb/data/matches')
ucl_path = Path(f'{matches_path}/{ucl_id}/{ucl_season_id}.json')
with open(ucl_path) as m:
    ucl_matches = json.load(m)

ucl_2019_final = ucl_matches[0]
ucl_2019_final_id = ucl_2019_final['match_id']
print(f'UCL final match: {ucl_2019_final}\nUCL final id: {ucl_2019_final_id}')

UCL final match: {'match_id': 22912, 'match_date': '2019-06-01', 'kick_off': '21:00:00.000', 'competition': {'competition_id': 16, 'country_name': 'Europe', 'competition_name': 'Champions League'}, 'season': {'season_id': 4, 'season_name': '2018/2019'}, 'home_team': {'home_team_id': 38, 'home_team_name': 'Tottenham Hotspur', 'home_team_gender': 'male', 'home_team_group': None, 'country': {'id': 68, 'name': 'England'}, 'managers': [{'id': 81, 'name': 'Mauricio Roberto Pochettino Trossero', 'nickname': 'Mauricio Pochettino', 'dob': '1972-03-02', 'country': {'id': 11, 'name': 'Argentina'}}]}, 'away_team': {'away_team_id': 24, 'away_team_name': 'Liverpool', 'away_team_gender': 'male', 'away_team_group': None, 'country': {'id': 68, 'name': 'England'}, 'managers': [{'id': 94, 'name': 'Jürgen Klopp', 'nickname': None, 'dob': '1967-06-16', 'country': {'id': 85, 'name': 'Germany'}}]}, 'home_score': 0, 'away_score': 2, 'match_status': 'available', 'match_status_360': 'scheduled', 'last_updated':

In [6]:
# Explore the events of the match based on the match ID
events_path = Path('../../data/raw/statsbomb/data/events')
ucl_2019_final_path = Path(f'{events_path}/{ucl_2019_final_id}.json')
with open(ucl_2019_final_path) as m:
    ucl_final_event = json.load(m)

print(f"Total events in the final: {len(ucl_final_event)}")
# print(ucl_final_event[0])


event_names = set()
for event in ucl_final_event:
    event_names.add((event['type']['id'], event['type']['name']))


print(event_names)
#ucl_df = pl.DataFrame(ucl_final_event)
#print(ucl_df.head(10))

Total events in the final: 3165
{(33, '50/50'), (9, 'Clearance'), (17, 'Pressure'), (43, 'Carry'), (4, 'Duel'), (14, 'Dribble'), (8, 'Offside'), (18, 'Half Start'), (2, 'Ball Recovery'), (40, 'Injury Stoppage'), (35, 'Starting XI'), (10, 'Interception'), (21, 'Foul Won'), (36, 'Tactical Shift'), (41, 'Referee Ball-Drop'), (30, 'Pass'), (22, 'Foul Committed'), (28, 'Shield'), (39, 'Dribbled Past'), (34, 'Half End'), (38, 'Miscontrol'), (42, 'Ball Receipt*'), (23, 'Goal Keeper'), (3, 'Dispossessed'), (6, 'Block'), (19, 'Substitution'), (16, 'Shot')}


In [7]:
# Look at the events i deem are the most telling and see what makes them up
# Look at pass events and see what it looks like structurally

# Pass id variable
pass_id = ()
for event in event_names:
    if event[1] == 'Pass':
        pass_id = event
        break


# Get the information of the pass event
ucl_final_event_pass = []

for event in ucl_final_event:
    if event['type']['name'] == 'Pass':
        ucl_final_event_pass.append(event)

print(ucl_final_event_pass[0]['pass'])


print('====================================================================================================')

# print the outcomes for the passes to see if any are successful
# This is a check to see how passes are graded and to be implemented in the next cell
successful_pass_count = 0
incomplete_pass_count = 0

# set for constant look up times
incomplete_set = {'Incomplete','Unknown','Out','Pass Offside', 'Injury Clearance'}

for passes in ucl_final_event_pass:
    if 'outcome' in passes['pass']:
        if passes['pass']['outcome']['name'] in incomplete_set:
            incomplete_pass_count += 1
        else:
            print(passes['pass']['outcome']['name'])
    else:
        successful_pass_count += 1

print(f"Successful pass count:{successful_pass_count}\nIncomplete pass count:{incomplete_pass_count}")

{'recipient': {'id': 3502, 'name': 'Joël Andre Job Matip'}, 'length': 27.252338, 'angle': 3.005404, 'height': {'id': 1, 'name': 'Ground Pass'}, 'end_location': [34.0, 43.8], 'body_part': {'id': 40, 'name': 'Right Foot'}, 'type': {'id': 65, 'name': 'Kick Off'}}
Successful pass count:649
Incomplete pass count:241


In [8]:
# Get a list of every player that:
# made a pass and the information there

# player_keys = ['player_id' ,'name','total_passes','successful_passes', 'avg_pass_length','avg_pass_angle']
player_pass_metrics = {}

#  set for constant look up times
incomplete_set = {'Incomplete','Unknown','Out','Pass Offside', 'Injury Clearance'}

def successful_pass_check(pass_event: dict):
    if 'outcome' not in pass_event['pass']:
        return 1
    else:
        return 0


# Go through entire event as i need the player information not just pass information
for event in ucl_final_event:
    # skip the events that are not passes
    if event['type']['name'] == 'Pass':
        p_id = event['player']['id']
        # Check if there is anything in player pass metrics
        if p_id not in player_pass_metrics:

            player_passes = {
                'player_id': p_id,
                'name': event['player']['name'],
                'total_passes': 1,
                'successful_passes': successful_pass_check(event),
                'avg_pass_length': event['pass']['length'],
                'avg_pass_angle': event['pass']['angle']
            }

            player_pass_metrics[p_id] = player_passes

        else:
                 # If player is already in the dict then update metrics
                # if event['player']['id'] in player_pass_metrics:
                    # Go to that entry
                    current_player = player_pass_metrics[p_id]
                    # record the previous count fo calculations later
                    previous_pass_total = current_player['total_passes']
                    current_player['total_passes'] += 1
                    # Go into the event find the successful and unsuccessful ones
                    current_player['successful_passes'] += successful_pass_check(event)

                    # if 'outcome' in event['pass']:
                    #     if event['pass']['outcome']['name'] not in incomplete_set:
                    #         player['successful_passes'] += 1


                    # Calculate the average pass len
                    # Multiply the avg by the old count to get sum
                    prev_avg_sum = current_player['avg_pass_length'] * previous_pass_total
                    # Add new length to the prev sum
                    curr_avg_sum = prev_avg_sum + event['pass']['length']
                    # Update the average
                    current_player['avg_pass_length']= curr_avg_sum / current_player['total_passes']

                    # Calc the average pass angle same formula as above
                    prev_angle_sum = current_player['avg_pass_angle'] * previous_pass_total
                    curr_angle_sum = prev_angle_sum + event['pass']['angle']
                    current_player['avg_pass_angle'] = curr_angle_sum / current_player['total_passes']



print(next(iter(player_pass_metrics.values())))

{'player_id': 3532, 'name': 'Jordan Brian Henderson', 'total_passes': 26, 'successful_passes': 16, 'avg_pass_length': 21.17902310769231, 'avg_pass_angle': 0.0212693207076923}


In [9]:
# Further tests
total = sum(p['total_passes'] for p in player_pass_metrics.values())
print(f"Total passes across all players: {total}")
print(f"Total players who passed: {len(player_pass_metrics)}")

Total passes across all players: 890
Total players who passed: 28


In [10]:
passes_under_pressure = 0
for event in ucl_final_event:
    if event['type']['name'] == 'Pass':
        if event.get('under_pressure'):
            passes_under_pressure += 1

print(passes_under_pressure)

204


Carries Data Check ID 43

In [11]:
# Carry id variable
carry_id = ()
for event in event_names:
    if event[1] == 'Carry':
        carry_id = event
        break

# print(carry_id)

ucl_final_event_carries = []

for event in ucl_final_event:
    if event['type']['name'] == 'Carry':
        ucl_final_event_carries.append(event)

print(f'Number of carries in UCL final:{len(ucl_final_event_carries)}\n:{ucl_final_event_carries[0]}')

Number of carries in UCL final:669
:{'id': '609587a5-193c-469b-be3c-63131c646e59', 'index': 7, 'period': 1, 'timestamp': '00:00:01.875', 'minute': 0, 'second': 1, 'type': {'id': 43, 'name': 'Carry'}, 'possession': 2, 'possession_team': {'id': 24, 'name': 'Liverpool'}, 'play_pattern': {'id': 9, 'name': 'From Kick Off'}, 'team': {'id': 24, 'name': 'Liverpool'}, 'player': {'id': 3502, 'name': 'Joël Andre Job Matip'}, 'position': {'id': 3, 'name': 'Right Center Back'}, 'location': [34.0, 43.8], 'duration': 1.482954, 'related_events': ['10f8eed9-fa49-4170-9057-d9418e9546ae', 'c7297515-ae92-4fb4-93a8-59745b8dfa90'], 'carry': {'end_location': [36.1, 44.0]}}


In [12]:
carry_pressure_count = 0

for event in ucl_final_event:
    if event['type']['name'] == 'Carry':
        if event.get('under_pressure'):
            carry_pressure_count += 1

print(carry_pressure_count)


244


The actual StatsBomb pitch dimensions are standardised:

X: 0 to 120
Y: 0 to 80

StatsBomb pitch dimensions: 120 x 80
X axis: 0 = own goal, 120 = opponent goal
Y axis: 0 = left touchline, 80 = right touchline

SHOT EXTRACTION EXPLORATION

In [13]:
shot_outcomes = set()
for event in ucl_final_event:
    if event['type']['name'] == 'Shot':
        shot_outcomes.add(event['shot']['outcome']['name'])


shots_without_outcome = 0
for event in ucl_final_event:
    if event['type']['name'] == 'Shot':
        if 'outcome' not in event['shot']:
            shots_without_outcome += 1

print(f"Shots without outcome: {shots_without_outcome}")


print(shot_outcomes)

Shots without outcome: 0
{'Off T', 'Saved', 'Wayward', 'Blocked', 'Goal'}


PRESSURES EXPLORATION

In [14]:
outcome_set = set()

for event in ucl_final_event:
    if event['type']['name'] == 'Pressure':
        outcome = event.get('outcome',{})
        if outcome :
            outcome_set.add(outcome.get('name', 'Unknown'))

print(outcome_set)

set()


Interceptions

In [15]:
# Check the interceptions outcome types to correspond with
# success

interception_outcomes = set()

for event in ucl_final_event:
    if event['type']['name'] == 'Interception':
        outcome = event['interception']['outcome']
        if outcome:
            interception_outcomes.add(outcome['name'])


print(interception_outcomes)

{'Success In Play', 'Won', 'Lost Out', 'Lost In Play'}


Clearances

In [23]:
clearance_type = set()

for event in ucl_final_event:
    if event['type']['name'] == 'Clearance':
        clearance_type.add(event['clearance']['body_part']['name'])

print(clearance_type)




{'Right Foot', 'Other', 'Left Foot', 'Head'}


In [24]:
clearance_keys = set()
for event in ucl_final_event:
    if event['type']['name'] == 'Clearance':
        for key in event['clearance'].keys():
            clearance_keys.add(key)

print(clearance_keys)

{'other', 'aerial_won', 'head', 'left_foot', 'right_foot', 'body_part'}
